In [1]:
!rm -rf .five_minute_cache

# Fleche in Five Minutes

*A persistent cache for expensive Python functions — `lru_cache` on steroids.*

Decorate a function and `fleche` stores every result under a SHA256 key built from the
function's identity and the **content** of its arguments. Results survive restarts, can
live in files, HDF5, SQL, or on another machine — and everything you ever computed stays
queryable like a small database.

If your day involves functions that take minutes to days — structure relaxations,
phonons, MD, training runs — and you re-run them more often than you'd like, this is
for you.

```
pip install fleche          # or: conda install -c conda-forge fleche
```

## 1. Decorate and forget

Point the active cache at a directory (in real projects you'd do this once in a
`fleche.toml` next to your code — see the end of this notebook), then just decorate:

In [2]:
import time
from fleche import fleche, cache, tags, wrap_executor
from fleche.caches import Cache

cache(Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"}));

In [3]:
@fleche
def expensive(x):
    print(f"crunching {x} ...")
    time.sleep(2)                     # pretend this is a real calculation
    return x ** 2


start = time.time()
print(expensive(4), f"  <- first call: {time.time() - start:.2f} s")

start = time.time()
print(expensive(4), f"  <- second call: {time.time() - start:.4f} s")

crunching 4 ...


16   <- first call: 2.00 s
16   <- second call: 0.0009 s


The second call never enters the function body. Unlike `lru_cache`, the result is on
disk, not in process memory — restart the kernel (skip the cleanup cell at the top) and
it is still instant. A brand-new cache object pointed at the same directory already
knows the answer:

In [4]:
fresh = Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"})
with cache(fresh):
    print("cached?", expensive.fleche.contains(4))

cached? True


The cache is just a directory — `rsync` it, back it up, share it with your group.

## 2. Keys are content, not object identity

Arguments are digested by **value** — not `id()`, not pickle bytes. NumPy arrays,
pandas frames, dataclasses, and nested containers work out of the box; equal content
means equal key, no matter who constructed the object:

In [5]:
import numpy as np


@fleche
def norm(arr):
    print("computing ...")
    return float(np.linalg.norm(arr))


a = np.linspace(0, 1, 1_000_000)
print(norm(a))
print(norm(a.copy()))                 # different object, same content -> cache hit

computing ...
577.3504135273193
577.3504135273193


The digest machinery is pluggable for third-party types. If you use ASE:
`pip install fleche-ase` and `Atoms`, `Calculator`, and `VibrationsData` objects digest
correctly with **zero further setup** (registered via entry points):

In [6]:
try:
    from ase.build import bulk
    from ase.calculators.emt import EMT

    @fleche
    def energy(atoms):
        print("running EMT ...")
        atoms = atoms.copy()
        atoms.calc = EMT()
        return atoms.get_potential_energy()

    print(energy(bulk("Cu", cubic=True)))
    print(energy(bulk("Cu", cubic=True)))   # freshly built Atoms -> cache hit
except ImportError:
    print("pip install ase fleche-ase to run this cell")

running EMT ...
-0.022726045434316333
-0.022726045434316333


Methods work too, without special handling: `self` is just another argument, digested by
content like any other. Two instances carrying the same state share cache entries, and
mutating the state you care about invalidates them:

In [7]:
from dataclasses import dataclass


@dataclass
class Simulation:
    temperature: float
    steps: int = 10

    @fleche
    def run(self, label):
        print(f"  ...actually running {label} at {self.temperature} K")
        return self.temperature * self.steps


print(Simulation(300.0).run("heat"))
print(Simulation(300.0).run("heat"))    # a different object, same state -> hit
print(Simulation(400.0).run("heat"))    # different state -> miss

  ...actually running heat at 300.0 K
3000.0
3000.0
  ...actually running heat at 400.0 K
4000.0


### What goes into the key

Name, module, and arguments — but *not* the function body, so editing a function keeps
serving the old results. Opt in with `hash_code=True` and every source edit invalidates
its entries:

In [8]:
@fleche(hash_code=True)
def free_energy(T):
    return -1.5 * T                       # first version of the model


print(free_energy(300), "  cached?", free_energy.fleche.contains(300))


@fleche(hash_code=True)
def free_energy(T):                       # same name, corrected model
    return -1.5 * T - 0.002 * T ** 2


print("after editing the body, cached?", free_energy.fleche.contains(300))
print(free_energy(300))

-450.0   cached? True
after editing the body, cached? False
-630.0


This hashes *this* function's source, not the code it calls — a change in a helper or in a
dependency invalidates nothing. Where that matters, say so explicitly with
`@fleche(version=...)` and bump it; `ignore=` and `require=` control which arguments take
part in the key at all.

## 3. Your cache is a database

Every call is recorded — arguments, runtime, metadata — *separately* from the (possibly
heavy) result values, so you can browse what you computed without deserializing any of
it. Tag calls, then query into a pandas DataFrame:

In [9]:
@fleche
def relax(a, k):
    time.sleep(0.1)                   # pretend
    return {"energy": -a * k, "volume": a ** 3}


with tags(project="five-minute-demo"):
    for a_lat in (3.5, 3.6, 3.7):
        relax(a_lat, k=4)

relax.fleche.query().table(arguments=["a", "k"], results=True)

,name,module,result,timestart,timestop,walltime,project,a,k
ad2b,relax,__main__,"{'energy': -14.0, 'volume': 42.875}",2026-08-07 13:43:42.915848732-04:00,2026-08-07 13:43:43.016426086-04:00,0.100577,five-minute-demo,3.5,4
0275,relax,__main__,"{'energy': -14.4, 'volume': 46.656000000000006}",2026-08-07 13:43:43.017625809-04:00,2026-08-07 13:43:43.118124723-04:00,0.100499,five-minute-demo,3.6,4
ba58,relax,__main__,"{'energy': -14.8, 'volume': 50.653000000000006}",2026-08-07 13:43:43.119174004-04:00,2026-08-07 13:43:43.219753265-04:00,0.100579,five-minute-demo,3.7,4


Because that record is a real index, you can also ask what you *nearly* have — worth a
look before queueing a run someone already did at a slightly different parameter:

In [10]:
target = 3.68
close = (
    relax.fleche.query()
    .filter(lambda c: abs(c.arguments["a"] - target) < 0.1)
    .sorted(key=lambda c: abs(c.arguments["a"] - target))
)
print(f"nothing cached for a={target}, but {close.count()} nearby run(s):")
close.table(arguments=["a", "k"], results=True)

nothing cached for a=3.68, but 2 nearby run(s):


,name,module,result,timestart,timestop,walltime,project,a,k
ba58,relax,__main__,"{'energy': -14.8, 'volume': 50.653000000000006}",2026-08-07 13:43:43.119174004-04:00,2026-08-07 13:43:43.219753265-04:00,0.100579,five-minute-demo,3.7,4
0275,relax,__main__,"{'energy': -14.4, 'volume': 46.656000000000006}",2026-08-07 13:43:43.017625809-04:00,2026-08-07 13:43:43.118124723-04:00,0.100499,five-minute-demo,3.6,4


Queries chain (`filter`, `sorted`, `unique`, `groupby`, ...) and terminal methods can
`transfer()` matching entries to another cache or `evict()` them. And the view is not
per-function or per-project — `cache().table()` spans everything that ever wrote to this
cache:

In [11]:
cache().table()

,name,module,timestart,timestop,walltime,project
6bd4,expensive,__main__,2026-08-07 13:43:40.345532656-04:00,2026-08-07 13:43:42.345995426-04:00,2.000463,NaN
fcbd,norm,__main__,2026-08-07 13:43:42.366988659-04:00,2026-08-07 13:43:42.382326603-04:00,0.015338,NaN
31e0,energy,__main__,2026-08-07 13:43:42.879736662-04:00,2026-08-07 13:43:42.884329796-04:00,0.004593,NaN
7104,Simulation.run,__main__,2026-08-07 13:43:42.891786814-04:00,2026-08-07 13:43:42.897278547-04:00,0.005492,NaN
b439,Simulation.run,__main__,2026-08-07 13:43:42.898984671-04:00,2026-08-07 13:43:42.900375366-04:00,0.001391,NaN
cb7a,free_energy,__main__,2026-08-07 13:43:42.907274008-04:00,2026-08-07 13:43:42.907721281-04:00,0.000447,NaN
b823,free_energy,__main__,2026-08-07 13:43:42.910212040-04:00,2026-08-07 13:43:42.910451412-04:00,0.000239,NaN
ad2b,relax,__main__,2026-08-07 13:43:42.915848732-04:00,2026-08-07 13:43:43.016426086-04:00,0.100577,five-minute-demo
0275,relax,__main__,2026-08-07 13:43:43.017625809-04:00,2026-08-07 13:43:43.118124723-04:00,0.100499,five-minute-demo
ba58,relax,__main__,2026-08-07 13:43:43.119174004-04:00,2026-08-07 13:43:43.219753265-04:00,0.100579,five-minute-demo


## 4. Storing everything without filling the disk

Values are content-addressed, so equal results are stored once no matter how many calls
produced them. That is not a corner case — think relaxations converging to the same
minimum from different starting points:

In [12]:
from pathlib import Path


def cache_size(root="./.five_minute_cache"):
    return sum(f.stat().st_size for f in Path(root).rglob("*") if f.is_file())


@fleche
def relax_structure(scale):
    basin = round(scale)              # different starting points, same minimum
    return np.random.default_rng(basin).normal(size=(2000, 3))


before = cache_size()
scales = (0.9, 0.95, 1.0, 1.05, 1.1, 2.0)
raw = sum(relax_structure(s).nbytes for s in scales)
print(f"{len(scales)} results, {raw / 1024:.0f} KiB of arrays"
      f"  ->  cache grew by {(cache_size() - before) / 1024:.0f} KiB")

6 results, 281 KiB of arrays  ->  cache grew by 97 KiB


A cache that only ever grows is a liability, so throwing things away is explicit and in
two steps: evicting call records leaves the values behind, and `gc()` then sweeps
whatever no remaining call points at. Values shared with calls you kept survive.

In [13]:
relax_structure.fleche.query().evict()               # drop those call records
print("value entries reclaimed by gc():", len(cache().gc()))
print("earlier results untouched:", relax.fleche.contains(3.5, k=4))

value entries reclaimed by gc(): 8
earlier results untouched: True


## 5. Plays well with executors — including executorlib

`wrap_executor` patches any `concurrent.futures`-style executor. Fleche-decorated
functions carry the active cache into worker processes automatically, and cache hits
come back as already-completed futures **without ever being submitted**:

In [14]:
try:
    from executorlib import SingleNodeExecutor as Executor
except ImportError:
    from concurrent.futures import ProcessPoolExecutor as Executor


@fleche
def md_step(x):
    time.sleep(1)                     # pretend
    return x ** 3


for attempt in ("cold", "warm"):
    start = time.time()
    with Executor(max_workers=4) as ex:
        wrap_executor(ex)
        results = [f.result() for f in [ex.submit(md_step, x) for x in range(4)]]
    print(f"{attempt}: {results} in {time.time() - start:.2f} s")

cold: [0, 1, 8, 27] in 1.66 s
warm: [0, 1, 8, 27] in 0.02 s


On the warm pass every future is done before `submit` returns — the executor never sees
the work. Executor-specific kwargs like executorlib's `resource_dict` are forwarded
transparently.

## 6. Things worth a second look

- **Configuration by file** — drop a `fleche.toml` next to your project (or in `$HOME`);
  decorated code never changes:

  ```toml
  [default]
  cache = "persistent"

  [persistent]
  template = "cloudpickle"
  root = "~/.cache/fleche"
  ```

- **HDF5 storage** — `template = "bagofholding_hdf"` stores values via pyiron's
  [bagofholding](https://github.com/pyiron/bagofholding), for storage meant to outlive
  the environment that wrote it.
- **SQL call index** — keep call records in SQLite/Postgres for server-side filtering
  over large caches, while values stay in files or HDF5.
- **Caches on other machines** — `SshCache` forwards a whole cache over SSH to a
  `python -m fleche remote --serve` process on, say, your cluster's login node. Stack a
  local cache in front (`CacheStack`) for read-through with back-fill, or fan reads over
  teammates' caches with a read-only `CachePool`.
- **Take results home** — `query().transfer(other_cache)` moves selected results
  between caches (cluster → laptop); see `TransferWorkflow.ipynb`.
- **Ship a warm cache** — because a cache is just a directory, you can hand one to CI or
  to students so notebooks that would otherwise need a supercomputer replay instantly.
- **Inside pyiron_workflow** — `@fleche` and `@pyiron_workflow.atomic` do not interfere,
  and passing a fleche `Cache` to a `RunConfig` lets decorated nodes hit the same cache
  during a workflow run.
- **Signed storage** — HMAC-sign pickle-family entries with a `secret_key`; tampered or
  wrong-key entries surface as plain cache misses, never as executed code.

## Where to go next

- Docs: <https://fleche.readthedocs.io>
- Source: <https://github.com/pmrv/fleche>
- Deeper notebooks in this directory: `GettingStarted`, `ExtraMethods`,
  `StorageBackends`, `ConcurrentExecution`, `CacheStack`, `SecureStorage`,
  `TransferWorkflow`.